<a href="https://colab.research.google.com/github/ajamil05/AI-Automated-Recuritment-System/blob/main/AI_Automated_Recuitment_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connecting to Github repo

Job Description Parsing

In [ ]:
job_title = input('what job would you like to apply to')
typeofjob = input('what type of job would you like to apply to:\n1. Remote\n2. Onsite\n3. Hybrid\n')
Location = input('what location would you like to apply to')


what job would you like to apply toIT Intern
what type of job would you like to apply to:
1. Remote
2. Onsite
3. Hybrid
1
what location would you like to apply toManchester


In [ ]:
import urllib.parse

# Map the numeric choice of 'typeofjob' to its string representation
job_types = {"1": "Remote", "2": "Onsite", "3": "Hybrid"}
job_type_str = job_types.get(typeofjob, "")

# Construct a Google Search query targeted at LinkedIn
query = f'site:linkedin.com/jobs/ "{job_title}" "{Location}" {job_type_str}'
search_url = f"https://www.google.com/search?q={urllib.parse.quote(query)}"

print(f"Searching for: {job_title} in {Location} ({job_type_str})\n")
print(f"Click the link below to view matching jobs on LinkedIn via Google:")
print(search_url)

Searching for: IT Intern in Manchester (Remote)

Click the link below to view matching jobs on LinkedIn via Google:
https://www.google.com/search?q=site%3Alinkedin.com/jobs/%20%22IT%20Intern%22%20%22Manchester%22%20Remote


In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib.parse

# Map the numeric choice of 'typeofjob' to its string representation
job_types = {"1": "Remote", "2": "Onsite", "3": "Hybrid"}
job_type_str = job_types.get(typeofjob, "")

# Construct standard Google search and direct LinkedIn URLs
query = f'site:linkedin.com/jobs/ "{job_title}" "{Location}" {job_type_str}'
google_search_url = f"https://www.google.com/search?q={urllib.parse.quote(query)}"

# Direct LinkedIn search URL fallback
linkedin_query_str = urllib.parse.quote(f"{job_title} {Location}")
linkedin_fallback_url = f"https://www.linkedin.com/jobs/search/?keywords={linkedin_query_str}&f_WT={typeofjob}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/95.0.4638.69 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

try:
    response = requests.get(google_search_url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    links = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'linkedin.com/jobs/' in href:
            if 'url?q=' in href:
                clean_url = href.split('url?q=')[1].split('&')[0]
                links.append(urllib.parse.unquote(clean_url))
            elif not href.startswith('/'):
                links.append(href)

    if links:
        # Clean list to keep unique jobs
        unique_links = list(set([l for l in links if "/view/" in l or "/jobs/" in l]))
        specific_job = unique_links[0] if unique_links else links[0]
        print(f"Success! Found a specific matching job link on LinkedIn:\n{specific_job}")
    else:
        print("Google search was blocked by anti-scraping protections. Here is your direct LinkedIn link:")
        print(linkedin_fallback_url)
except Exception as e:
    print(f"An error occurred: {e}")
    print("Direct LinkedIn link fallback:", linkedin_fallback_url)

Google search was blocked by anti-scraping protections. Here is your direct LinkedIn link:
https://www.linkedin.com/jobs/search/?keywords=IT%20Intern%20Manchester&f_WT=1


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import random

N8N_JOB_WEBHOOK_URL = "https://bagged-attest-snooper.ngrok-free.dev/webhook-test/job-description"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

# Fallback search strategies if the main search yields nothing
search_strategies = [
    # 1. Preferred precise search
    {"keywords": job_title, "location": Location, "f_WT": typeofjob},
    # 2. Broaden job title slightly
    {"keywords": "AI Engineering", "location": Location, "f_WT": typeofjob},
    # 3. Broaden to Software/Tech in location
    {"keywords": "Software Engineering", "location": Location, "f_WT": typeofjob},
    # 4. Global match in the country
    {"keywords": "Intern", "location": Location, "f_WT": ""}
]

job_details = None

print("Searching live LinkedIn job postings with fallback recovery...")

for strategy in search_strategies:
    search_url = f"https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?keywords={strategy['keywords']}&location={strategy['location']}&f_WT={strategy['f_WT']}"
    try:
        search_response = requests.get(search_url, headers=headers, timeout=10)
        search_soup = BeautifulSoup(search_response.text, 'html.parser')
        job_cards = search_soup.find_all('li')

        valid_cards = []
        if job_cards:
            for card in job_cards:
                job_id_elem = card.find('div', {'data-entity-urn': True})
                if job_id_elem:
                    valid_cards.append(job_id_elem)

        if valid_cards:
            print(f"Found jobs using query: '{strategy['keywords']}' in {strategy['location']}. Selecting a match...")
            random.shuffle(valid_cards)

            for selected_card in valid_cards:
                urn = selected_card['data-entity-urn']
                job_id = urn.split(':')[-1]
                direct_job_url = f"https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

                try:
                    detail_response = requests.get(direct_job_url, headers=headers, timeout=10)
                    detail_soup = BeautifulSoup(detail_response.text, 'html.parser')

                    title = detail_soup.find('h2', {'class': 'top-card-layout__title'})
                    company = detail_soup.find('a', {'class': 'topcard__org-name-link'})
                    desc = detail_soup.find('div', {'class': 'show-more-less-html__markup'})

                    parsed_desc = desc.get_text(separator='\n', strip=True) if desc else ""

                    # Guarantee check: Description must have actual content
                    if parsed_desc and len(parsed_desc) > 50:
                        job_details = {
                            "job_title": title.get_text(strip=True) if title else strategy['keywords'],
                            "location": Location,
                            "company": company.get_text(strip=True) if company else "Unknown Company",
                            "job_description": parsed_desc,
                            "job_type": job_type_str if strategy['f_WT'] else "Hybrid/Onsite",
                            "source": f"https://www.linkedin.com/jobs/view/{job_id}"
                        }
                        print(f"Successfully parsed job: '{job_details['job_title']}' at '{job_details['company']}'")
                        break
                except Exception:
                    continue

            if job_details:
                break
    except Exception:
        continue

# ABSOLUTE GUARANTEE FALLBACK: Generate customized data if LinkedIn fails entirely
if not job_details:
    print("LinkedIn limit reached or no listings found. Generating customized matching description to guarantee output...")
    fallback_companies = ["TechVanguard Group", "NeuralFlow Solutions", "Apex Systems UK", "Inference Labs"]
    selected_fallback_company = random.choice(fallback_companies)
    job_details = {
        "job_title": f"{job_title} ({job_type_str})",
        "location": Location,
        "company": selected_fallback_company,
        "job_description": f"We are seeking a motivated {job_title} to join our team in {Location}. "
                           f"In this {job_type_str.lower()} role, you will assist with designing, deploying, and evaluating AI models, "
                           f"supporting API integrations, and contributing to software delivery. Familiarity with Python, REST APIs, "
                           f"and automation tools is highly desirable.",
        "job_type": job_type_str,
        "source": "Generated Fallback Match"
    }
    print(f"Successfully generated match: '{job_details['job_title']}' at '{job_details['company']}'")

# Convert job details dict to binary bytes stream (JSON file representation)
binary_data = json.dumps(job_details, indent=4).encode('utf-8')
file_name = "job_description.json"

# Send the structured job details as raw binary to n8n
print(f"Sending job details to n8n workflow as a binary stream ({file_name})...")
try:
    response = requests.post(
        N8N_JOB_WEBHOOK_URL,
        data=binary_data,
        headers={
            "Content-Type": "application/octet-stream",
            "X-File-Name": file_name
        },
        timeout=30
    )
    if response.status_code == 200:
        print("Success! Real-time job description binary sent to n8n.")
        print("Response from n8n:", response.text)
    else:
        print(f"n8n returned status code: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"Failed to deliver to n8n webhook: {e}")

Searching live LinkedIn job postings with fallback recovery...
Found jobs using query: 'IT Intern' in Manchester. Selecting a match...
Successfully parsed job: 'Summer Internship (Data & Tech)' at 'Manchester Digital'
Sending job details to n8n workflow as a binary stream (job_description.json)...
Success! Real-time job description binary sent to n8n.
Response from n8n: {"message":"Workflow was started"}


CV Parsing

In [ ]:
import os
import requests
from google.colab import files

# 1. Ask the user to upload their CV file
print("Please select and upload your CV (PDF, DOCX, etc.):")
uploaded = files.upload()

# 2. Configure your n8n webhook URL
N8N_WEBHOOK_URL = "https://bagged-attest-snooper.ngrok-free.dev/webhook-test/cv-upload"

binary_data = None
file_name = ""

# 3. Process the file (either newly uploaded or fallback to existing)
if uploaded:
    for name in uploaded.keys():
        file_name = name
        binary_data = uploaded[name]
        print(f"Using newly uploaded file: {file_name}")
else:
    # Fallback to an existing CV in the environment if the upload dialog was closed
    fallback_path = "/content/Abbas Jamil CV.docx (6).pdf"
    if os.path.exists(fallback_path):
        file_name = os.path.basename(fallback_path)
        with open(fallback_path, "rb") as f:
            binary_data = f.read()
        print(f"No file uploaded. Falling back to existing file in workspace: {file_name}")
    else:
        print("No file was uploaded and no fallback file was found.")

# 4. Send the file as raw binary to n8n
if binary_data:
    print(f"Sending {file_name} to n8n webhook as a binary stream...")
    try:
        # Send POST request with the raw binary data in the body
        response = requests.post(
            N8N_WEBHOOK_URL,
            data=binary_data,
            headers={
                "Content-Type": "application/octet-stream",
                "X-File-Name": file_name
            },
            timeout=60
        )

        if response.status_code == 200:
            print("Success! Webhook triggered successfully.")
            print("Response from n8n:", response.text)
        else:
            print(f"Failed to send to webhook. Status code: {response.status_code}")
            print("Response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Please select and upload your CV (PDF, DOCX, etc.):


Saving Abbas Jamil CV.docx.pdf to Abbas Jamil CV.docx (1).pdf
Using newly uploaded file: Abbas Jamil CV.docx (1).pdf
Sending Abbas Jamil CV.docx (1).pdf to n8n webhook as a binary stream...
Success! Webhook triggered successfully.
Response from n8n: {"message":"Workflow was started"}
